# Target Transformation: raw vs log1p

`selling_price`의 오른쪽 꼬리가 긴 분포를 확인한 뒤, 로그 변환이 실제로 도움이 되는지 가정하지 않고 train 내부 5-Fold Cross Validation으로 비교합니다.

고정 test split은 이전 baseline 비교에 이미 사용했으므로 이번 비교에서는 예측·평가·후보 선택에 사용하지 않습니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)


pandas: 3.0.5
NumPy: 2.5.3
scikit-learn: 1.9.1


## Method

`02_preprocessing.ipynb`과 동일하게 중복 제거, 차량 제원 숫자 변환, 가격 구간 기반 80:20 split을 재현합니다. 각 CV fold에서는 training subset으로만 결측 대체·indicator·`name_prefix_2`·One-Hot Encoding을 학습합니다. Ridge에는 동일한 7개 수치형 feature scaling을 적용하고 RandomForest에는 scaling하지 않습니다.

모델과 5-Fold 설정은 그대로 유지하고 target 표현만 raw와 `log1p`로 바꿉니다. 모든 지표는 `expm1` 역변환 후 원래 가격 단위에서 계산합니다.


In [2]:
data_path = Path("../data/raw/Car details v3.csv")
df = pd.read_csv(data_path)
df_clean = df.drop_duplicates().copy()
df_preprocessed = df_clean.copy()

for column in ["mileage", "engine", "max_power"]:
    df_preprocessed[column] = pd.to_numeric(
        df_preprocessed[column].str.extract(r"^([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce",
    )

df_preprocessed.loc[df_preprocessed["mileage"].eq(0), "mileage"] = pd.NA
df_preprocessed.loc[df_preprocessed["max_power"].eq(0), "max_power"] = pd.NA

torque_text = df_preprocessed["torque"].str.strip()
torque_lower = torque_text.str.lower()
torque_first_number = pd.to_numeric(
    torque_text.str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
    errors="coerce",
)
nm_mask = torque_lower.str.contains("nm", regex=False, na=False)
kgm_mask = torque_lower.str.contains("kgm", regex=False, na=False)
nm_only_mask = nm_mask & ~kgm_mask
kgm_only_mask = kgm_mask & ~nm_mask
both_mask = nm_mask & kgm_mask
no_unit_mask = ~nm_mask & ~kgm_mask & torque_text.notna()

torque_nm = pd.Series(np.nan, index=df_preprocessed.index, dtype="float64")
torque_nm.loc[nm_only_mask] = torque_first_number.loc[nm_only_mask]
torque_nm.loc[kgm_only_mask & torque_first_number.lt(50)] = (
    torque_first_number.loc[kgm_only_mask & torque_first_number.lt(50)] * 9.80665
)
torque_nm.loc[both_mask] = torque_first_number.loc[both_mask]
torque_nm.loc[torque_text.eq("789Nm@ 2250rpm")] = np.nan

bmw_kgm_strings = [
    "51@ 1,750-3,000(kgm@ rpm)",
    "53@ 2,000-2,750(kgm@ rpm)",
]
bmw_kgm_mask = torque_text.isin(bmw_kgm_strings)
torque_nm.loc[bmw_kgm_mask] = torque_first_number.loc[bmw_kgm_mask] * 9.80665
remaining_large_kgm_mask = kgm_only_mask & torque_first_number.ge(50) & ~bmw_kgm_mask
torque_nm.loc[remaining_large_kgm_mask] = torque_first_number.loc[remaining_large_kgm_mask]
torque_nm.loc[no_unit_mask] = torque_first_number.loc[no_unit_mask]
df_preprocessed["torque"] = torque_nm

price_bins = pd.cut(
    df_preprocessed["selling_price"],
    bins=[0, 200000, 400000, 600000, 1000000, float("inf")],
    labels=["0~199,999", "200,000~399,999", "400,000~599,999", "600,000~999,999", "1,000,000 이상"],
    right=False,
    include_lowest=True,
)
X = df_preprocessed.drop(columns=["selling_price"]).copy()
y = df_preprocessed["selling_price"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1385, random_state=42, stratify=price_bins
)

assert df_preprocessed.shape == (6926, 13)
assert X_train.shape == (5541, 12) and y_train.shape == (5541,)
assert X_train.index.equals(y_train.index)
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("고정 test는 분할만 재현했으며 이번 비교에 사용하지 않음:", X_test.shape, y_test.shape)


X_train: (5541, 12)
y_train: (5541,)
고정 test는 분할만 재현했으며 이번 비교에 사용하지 않음: (1385, 12) (1385,)


In [3]:
raw_target_stats = pd.Series({
    "count": y_train.count(),
    "mean": y_train.mean(),
    "median": y_train.median(),
    "min": y_train.min(),
    "max": y_train.max(),
    "skewness": y_train.skew(),
})
log_target = np.log1p(y_train)
log_target_stats = pd.Series({
    "count": log_target.count(),
    "mean": log_target.mean(),
    "median": log_target.median(),
    "min": log_target.min(),
    "max": log_target.max(),
    "skewness": log_target.skew(),
})
target_stats = pd.DataFrame({"raw": raw_target_stats, "log1p": log_target_stats})
print(target_stats.to_string())


                   raw        log1p
count     5.541000e+03  5541.000000
mean      5.180143e+05    12.865035
median    4.050000e+05    12.911645
min       2.999900e+04    10.308953
max       1.000000e+07    16.118096
skewness  5.783779e+00    -0.138456


In [4]:
def make_name_prefix_2(frame):
    result = frame.copy()
    result["name_prefix_2"] = result["name"].str.split().str[:2].str.join(" ")
    return result.drop(columns=["name"])


name_prefix_transformer = FunctionTransformer(make_name_prefix_2, validate=False)
base_numeric_cols = ["year", "km_driven"]
continuous_spec_cols = ["mileage", "engine", "max_power", "torque"]
seats_col = ["seats"]
categorical_cols = ["name_prefix_2", "fuel", "seller_type", "transmission", "owner"]

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("base_numeric", "passthrough", base_numeric_cols),
        ("continuous_specs", SimpleImputer(strategy="median"), continuous_spec_cols),
        ("continuous_indicators", MissingIndicator(features="all"), continuous_spec_cols),
        ("seats", SimpleImputer(strategy="most_frequent"), seats_col),
        ("seats_indicator", MissingIndicator(features="all"), seats_col),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("base_numeric", StandardScaler(), base_numeric_cols),
        ("continuous_specs", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), continuous_spec_cols),
        ("continuous_indicators", MissingIndicator(features="all"), continuous_spec_cols),
        ("seats", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("scaler", StandardScaler()),
        ]), seats_col),
        ("seats_indicator", MissingIndicator(features="all"), seats_col),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

rf_model_pipeline = Pipeline([
    ("preprocessing", Pipeline([
        ("name_prefix", clone(name_prefix_transformer)),
        ("preprocessor", clone(rf_preprocessor)),
    ])),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
])
ridge_model_pipeline = Pipeline([
    ("preprocessing", Pipeline([
        ("name_prefix", clone(name_prefix_transformer)),
        ("preprocessor", clone(ridge_preprocessor)),
    ])),
    ("model", Ridge(alpha=1.0)),
])
print("Pipeline 후보 준비 완료: Ridge, RandomForest")


Pipeline 후보 준비 완료: Ridge, RandomForest


In [5]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(cv.split(X_train))

candidates = [
    ("Ridge", "raw", ridge_model_pipeline),
    ("Ridge", "log1p", TransformedTargetRegressor(
        regressor=clone(ridge_model_pipeline), func=np.log1p, inverse_func=np.expm1
    )),
    ("RandomForest", "raw", rf_model_pipeline),
    ("RandomForest", "log1p", TransformedTargetRegressor(
        regressor=clone(rf_model_pipeline), func=np.log1p, inverse_func=np.expm1
    )),
]
print("CV:", cv)
print("후보 수:", len(candidates))


CV: KFold(n_splits=5, random_state=42, shuffle=True)
후보 수: 4


In [6]:
def evaluate_candidate(model_name, target_representation, estimator):
    oof_prediction = pd.Series(np.nan, index=y_train.index, dtype="float64")
    validation_coverage = pd.Series(0, index=y_train.index, dtype="int64")
    fold_rows = []

    for fold, (train_positions, validation_positions) in enumerate(fold_splits, start=1):
        fold_estimator = clone(estimator)
        fold_estimator.fit(
            X_train.iloc[train_positions],
            y_train.iloc[train_positions],
        )
        prediction = fold_estimator.predict(X_train.iloc[validation_positions])
        actual = y_train.iloc[validation_positions]
        oof_prediction.iloc[validation_positions] = prediction
        validation_coverage.iloc[validation_positions] += 1
        fold_rows.append({
            "model": model_name,
            "target_representation": target_representation,
            "fold": fold,
            "mae": mean_absolute_error(actual, prediction),
            "rmse": root_mean_squared_error(actual, prediction),
            "r2": r2_score(actual, prediction),
        })

    assert oof_prediction.isna().sum() == 0
    assert validation_coverage.eq(1).all()
    return pd.DataFrame(fold_rows), oof_prediction, validation_coverage


fold_results = []
oof_predictions = {}
coverage_results = {}
for model_name, target_representation, estimator in candidates:
    key = f"{model_name}_{target_representation}"
    candidate_folds, candidate_oof, candidate_coverage = evaluate_candidate(
        model_name, target_representation, estimator
    )
    fold_results.append(candidate_folds)
    oof_predictions[key] = candidate_oof
    coverage_results[key] = candidate_coverage
    print("완료:", key)

fold_metrics = pd.concat(fold_results, ignore_index=True)
print("candidate-fold 결과 수:", len(fold_metrics))


완료: Ridge_raw
완료: Ridge_log1p
완료: RandomForest_raw
완료: RandomForest_log1p
candidate-fold 결과 수: 20


In [7]:
summary_rows = []
for (model_name, target_representation), group in fold_metrics.groupby(
    ["model", "target_representation"], sort=False
):
    key = f"{model_name}_{target_representation}"
    predictions = oof_predictions[key]
    negative_count = int((predictions < 0).sum())
    summary_rows.append({
        "model": model_name,
        "target_representation": target_representation,
        "cv_mae_mean": group["mae"].mean(),
        "cv_mae_std": group["mae"].std(ddof=0),
        "cv_rmse_mean": group["rmse"].mean(),
        "cv_rmse_std": group["rmse"].std(ddof=0),
        "cv_r2_mean": group["r2"].mean(),
        "cv_r2_std": group["r2"].std(ddof=0),
        "oof_negative_prediction_count": negative_count,
        "oof_negative_prediction_pct": negative_count / len(predictions) * 100,
    })

cv_summary = pd.DataFrame(summary_rows)
print(cv_summary.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


       model target_representation   cv_mae_mean  cv_mae_std  cv_rmse_mean  cv_rmse_std  cv_r2_mean  cv_r2_std  oof_negative_prediction_count  oof_negative_prediction_pct
       Ridge                   raw 124406.465627 3727.913627 245443.803774 31806.307755    0.780487   0.033710                            234                     4.223064
       Ridge                 log1p  83323.839206 3516.986113 189720.989487 15915.812404    0.867377   0.020256                              0                     0.000000
RandomForest                   raw  78457.889024 3867.836060 164896.704467 28340.606040    0.898948   0.027368                              0                     0.000000
RandomForest                 log1p  78161.849416 4130.050820 171738.880502 31731.176789    0.889622   0.036100                              0                     0.000000


In [8]:
expected_raw = {
    "Ridge": {"mae": 124406.47, "rmse": 245443.80, "r2": 0.7805},
    "RandomForest": {"mae": 78457.89, "rmse": 164896.70, "r2": 0.8989},
}
reproduction_rows = []
for model_name, expected in expected_raw.items():
    actual = cv_summary.query(
        "model == @model_name and target_representation == 'raw'"
    ).iloc[0]
    reproduction_rows.append({
        "model": model_name,
        "mae_difference": actual["cv_mae_mean"] - expected["mae"],
        "rmse_difference": actual["cv_rmse_mean"] - expected["rmse"],
        "r2_difference": actual["cv_r2_mean"] - expected["r2"],
    })

raw_reproduction = pd.DataFrame(reproduction_rows)
raw_reproduction_ok = bool(
    (raw_reproduction["mae_difference"].abs() < 0.02).all()
    and (raw_reproduction["rmse_difference"].abs() < 0.02).all()
    and (raw_reproduction["r2_difference"].abs() < 0.0001).all()
)
print(raw_reproduction.to_string(index=False, float_format=lambda value: f"{value:.8f}"))
print("기존 raw CV 결과 재현:", raw_reproduction_ok)


       model  mae_difference  rmse_difference  r2_difference
       Ridge     -0.00437286       0.00377445    -0.00001308
RandomForest     -0.00097644       0.00446684     0.00004811
기존 raw CV 결과 재현: True


In [9]:
comparison_rows = []
for model_name in ["Ridge", "RandomForest"]:
    raw = cv_summary.query(
        "model == @model_name and target_representation == 'raw'"
    ).iloc[0]
    log = cv_summary.query(
        "model == @model_name and target_representation == 'log1p'"
    ).iloc[0]
    raw_folds = fold_metrics.query(
        "model == @model_name and target_representation == 'raw'"
    ).sort_values("fold")
    log_folds = fold_metrics.query(
        "model == @model_name and target_representation == 'log1p'"
    ).sort_values("fold")
    comparison_rows.append({
        "model": model_name,
        "mae_raw_minus_log": raw["cv_mae_mean"] - log["cv_mae_mean"],
        "mae_change_pct": (raw["cv_mae_mean"] - log["cv_mae_mean"]) / raw["cv_mae_mean"] * 100,
        "rmse_raw_minus_log": raw["cv_rmse_mean"] - log["cv_rmse_mean"],
        "rmse_change_pct": (raw["cv_rmse_mean"] - log["cv_rmse_mean"]) / raw["cv_rmse_mean"] * 100,
        "r2_log_minus_raw": log["cv_r2_mean"] - raw["cv_r2_mean"],
        "mae_improved_folds": int((log_folds["mae"].to_numpy() < raw_folds["mae"].to_numpy()).sum()),
        "rmse_improved_folds": int((log_folds["rmse"].to_numpy() < raw_folds["rmse"].to_numpy()).sum()),
        "raw_negative_oof": int(cv_summary.query(
            "model == @model_name and target_representation == 'raw'"
        ).iloc[0]["oof_negative_prediction_count"]),
        "log_negative_oof": int(cv_summary.query(
            "model == @model_name and target_representation == 'log1p'"
        ).iloc[0]["oof_negative_prediction_count"]),
    })

target_comparison = pd.DataFrame(comparison_rows)
print(target_comparison.to_string(index=False, float_format=lambda value: f"{value:.6f}"))


       model  mae_raw_minus_log  mae_change_pct  rmse_raw_minus_log  rmse_change_pct  r2_log_minus_raw  mae_improved_folds  rmse_improved_folds  raw_negative_oof  log_negative_oof
       Ridge       41082.626421       33.022903        55722.814288        22.702881          0.086890                   5                    5               234                 0
RandomForest         296.039608        0.377323        -6842.176035        -4.149371         -0.009326                   3                    1                 0                 0


In [10]:
oof_integrity = pd.DataFrame({
    key: {
        "total": len(predictions),
        "missing": int(predictions.isna().sum()),
        "duplicate_validation_coverage": int((coverage_results[key] > 1).sum()),
        "uncovered": int((coverage_results[key] == 0).sum()),
    }
    for key, predictions in oof_predictions.items()
}).T

test_usage = pd.Series({
    "test_prediction": False,
    "test_metric": False,
    "test_based_selection": False,
})
print(oof_integrity.to_string())
print("\ntest usage")
print(test_usage.to_string())


                    total  missing  duplicate_validation_coverage  uncovered
Ridge_raw            5541        0                              0          0
Ridge_log1p          5541        0                              0          0
RandomForest_raw     5541        0                              0          0
RandomForest_log1p   5541        0                              0          0

test usage
test_prediction         False
test_metric             False
test_based_selection    False


## Result interpretation

기존 raw target CV 결과 재현 여부는 **True**입니다. 재현이 확인됐으므로 target 표현 비교를 해석합니다.

- Ridge log1p: MAE 개선, RMSE 개선, R² 개선
- RandomForest log1p: MAE 개선, RMSE 악화, R² 악화

이 결과는 동일한 train split과 5개 fold에서 target 표현만 바꾼 비교입니다. 통계적 유의성 검정을 수행하지 않았고, 고정 test 결과를 사용해 후보를 선택하지 않았습니다.
